In [0]:
import os
import requests
from pathlib import Path

# ----------------------------
# 1) Choose stations + years
# ----------------------------
STATIONS_BY_STATE = {
    "IL": {
        "Chicago_OHare": "USW00094846",
        "Chicago_Midway": "USW00014819",
        "Peoria": "USW00014842",
        "Moline_Quad_City": "USW00014923",
        "Rockford": "USW00094822",
        "Springfield": "USW00093822",
    },
    "IN": {
        "Indianapolis": "USW00093819",
        "Evansville": "USW00093817",
        "Fort_Wayne": "USW00014827",
        "South_Bend": "USW00014848",
        "Terre_Haute": "USW00093823",
        "Lafayette": "USW00014835",
    },
    "IA": {
        "Des_Moines": "USW00014933",
        "Sioux_City": "USW00014943",
        "Mason_City": "USW00014940",
        "Dubuque": "USW00094908",
        "Cedar_Rapids": "USW00014990",
        "Waterloo": "USW00094910",
    },
    "MN": {
        "Minneapolis_StPaul": "USW00014922",
        "International_Falls": "USW00014918",
        "Rochester": "USW00014925",
        "St_Cloud": "USW00014926",
        "St_Paul_Downtown": "USW00014927",
        "Flying_Cloud": "USW00094963",
    },
    "NE": {
        "Omaha": "USW00014942",
        "Lincoln": "USW00014939",
        "Grand_Island": "USW00014935",
        "Norfolk": "USW00014941",
    },
}

YEARS = list(range(2010, 2025))   # 2010 through 2024 inclusive

# ----------------------------
# 2) NOAA base URLs
# ----------------------------
LCD_BASE = "https://www.ncei.noaa.gov/oa/local-climatological-data/v2/access"
NORMALS_BASE = "https://www.ncei.noaa.gov/data/normals-annualseasonal/1991-2020/access"

# ----------------------------
# 3) DBFS output folders
# ----------------------------
lcd_dir = Path("/Volumes/workspace/default/databricks-hackathon/lcd")
normals_dir = Path("/Volumes/workspace/default/databricks-hackathon/normals")
lcd_dir.mkdir(parents=True, exist_ok=True)
normals_dir.mkdir(parents=True, exist_ok=True)

# ----------------------------
# 4) Download helper
# ----------------------------
def download_file(url: str, out_path: Path):
    try:
        r = requests.get(url, timeout=120)
        if r.status_code == 200:
            with open(out_path, "wb") as f:
                f.write(r.content)
            print(f"Downloaded: {out_path.name}")
            return True
        else:
            print(f"Missing: {url} -> HTTP {r.status_code}")
            return False
    except Exception as e:
        print(f"Error downloading {url}: {e}")
        return False

# ----------------------------
# 5) Download LCD files (2010-2024)
# ----------------------------
for state, stations in STATIONS_BY_STATE.items():
    for station_name, station_id in stations.items():
        for year in YEARS:
            url = f"{LCD_BASE}/{year}/LCD_{station_id}_{year}.csv"
            out_path = lcd_dir / f"{state}_{station_name}_{station_id}_{year}.csv"
            download_file(url, out_path)

# ----------------------------
# 6) Download normals files (fixed 1991-2020 normals)
# ----------------------------
for state, stations in STATIONS_BY_STATE.items():
    for station_name, station_id in stations.items():
        url = f"{NORMALS_BASE}/{station_id}.csv"
        out_path = normals_dir / f"{state}_{station_name}_{station_id}_normals.csv"
        download_file(url, out_path)